# zero-grad-set-none — worked example 2: Grad reallocates after zeroing

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `zero-grad-set-none`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Setting `.grad = None` does not break training: the very next `.backward()` allocates a fresh gradient tensor from scratch. Demonstrating this round-trip — grad present, set to None, backward, grad present again — confirms the None convention is functionally equivalent to a zeroed buffer.

## Worked solution

We show the None-then-backward round trip on a scalar parameter.

1. We create a leaf parameter `w` and do one backward so `w.grad` is populated.
2. We set `w.grad = None` (our zero_grad), confirming it is cleared.
3. We run a second backward on a fresh loss; PyTorch allocates a new `w.grad` automatically.

We print the grad at each stage to show it goes present → None → present, proving the convention loses nothing.

In [ ]:
import torch as t

w = t.tensor([2.0], requires_grad=True)
(w ** 2).sum().backward()
grad1 = w.grad.clone()

w.grad = None
cleared = w.grad is None

(3 * w).sum().backward()
grad2 = w.grad.clone()

print('grad after first backward:', grad1.tolist())
print('cleared to None:', cleared)
print('grad after second backward:', grad2.tolist())